In [1]:
from tensorflow.keras.layers import Input, Conv2D, Dense, ReLU, Flatten, Softmax
from tensorflow.keras import Model
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np
import matplotlib.pyplot as plt

In [2]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [3]:
# Convert y_train into one-hot format
temp = []
for i in range(len(y_train)):
    temp.append(to_categorical(y_train[i], num_classes=10))
y_train = np.array(temp)
# Convert y_test into one-hot format
temp = []
for i in range(len(y_test)):    
    temp.append(to_categorical(y_test[i], num_classes=10))
y_test = np.array(temp)

In [4]:
#reshaping
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1)

In [5]:
inputs = Input(shape=(28,28,1))
out = Conv2D(1,3)(inputs)
out = ReLU()(out)
out = Flatten()(out)
out = Dense(10, activation=None)(out)
out = Softmax()(out)
model = Model(inputs, out)

In [6]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 1)      │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 26, 26, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 676)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         6,770 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,780 (26.48 KB)

 Trainable params: 6,780 (26.48 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['acc']
    )

In [8]:
model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - acc: 0.8134 - loss: 3.2269 - val_acc: 0.8628 - val_loss: 0.5466
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.8982 - loss: 0.3801 - val_acc: 0.9161 - val_loss: 0.2952
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9146 - loss: 0.2977 - val_acc: 0.9234 - val_loss: 0.2663
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9186 - loss: 0.2819 - val_acc: 0.9223 - val_loss: 0.2590
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9226 - loss: 0.2662 - val_acc: 0.9291 - val_loss: 0.2357
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9257 - loss: 0.2488 - val_acc: 0.9341 - val_loss: 0.2236
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9292 - loss: 0.2342 - val_acc: 0.9324 - val_loss: 0.2159
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - acc: 0.9294 - loss: 0.2320 - val_acc: 0.9293 - val_loss: 0.2319
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━

In [9]:
X = X_test[0]
X.shape

(28, 28, 1)

In [10]:
model2 = Model(model.input, model.layers[-2].output)

In [11]:
y = model2.predict(X_test[[0]]) - model.weights[3].numpy()
y

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


array([[-12.809172 , -17.94385  ,  -1.4110506,   2.5295072, -21.79472  ,
         -6.7403245, -25.659609 ,  11.96709  ,  -1.2282121,  -6.798327 ]],
      dtype=float32)

In [22]:
in_json = {
    "in": X.astype(int).astype(str).tolist(),
    "conv2d_weights": (model.weights[0].numpy()*(10**9)).round().astype(int).astype(str).tolist(),
    "conv2d_bias": (model.weights[1].numpy()*(10**9)).round().astype(int).astype(str).tolist(), # no need to sqaure the scaling because input was not scaled
    "dense_weights":(model.weights[2].numpy()*(10**9)).round().astype(int).astype(str).tolist(),
    "dense_bias": np.zeros(model.weights[3].numpy().shape).astype(int).astype(str).tolist() # zero because we are not doing softmax in circom, just argmax
}

In [23]:
out_json = {
    "scale": 10**-18,
    "out": y.flatten().tolist(),
    "label": int(y.argmax())
}
out_json

{'scale': 1e-18,
 'out': [-12.809171676635742,
  -17.943849563598633,
  -1.41105055809021,
  2.5295071601867676,
  -21.794719696044922,
  -6.7403244972229,
  -25.659608840942383,
  11.967089653015137,
  -1.2282121181488037,
  -6.7983269691467285],
 'label': 7}

In [24]:
import json

In [25]:
with open("mnist_input.json", "w") as f:
    json.dump(in_json, f)

In [26]:
with open("mnist_output.json", "w") as f:
    json.dump(out_json, f)